# Clase 26 - PySpark: Arquitectura de Spark y primeros DataFrames

### Celda de configuración común
Ejecuta siempre esta celda primero (y de nuevo tras reiniciar el kernel).

In [1]:
import os
from pathlib import Path

# 1. Detectar el entorno
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    ENTORNO = "databricks"
elif Path("/opt/datos").is_dir():
    ENTORNO = "docker"            # contenedor jupyter del Tutorial 2
else:
    ENTORNO = "local"             # Spark local del Tutorial 1

# 2. SparkSession: en Databricks ya existe; en los otros dos se crea
if ENTORNO != "databricks":
    from pyspark.sql import SparkSession

    builder = SparkSession.builder.appName("clase_26")
    if ENTORNO == "local":
        builder = builder.master("local[*]")
        if os.name == "nt":       # Windows: caché de JARs aislada del Tutorial 1
            builder = builder.config("spark.jars.ivy", "C:/SparkLocal/tools/cache/ivy2")
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")                     # menos mensajes en la consola
    spark.conf.set("spark.sql.repl.eagerEval.enabled", True)    # display(df) dibuja una tabla

# 3. Carpeta de datos de la clase
if ENTORNO == "databricks":
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.clase_26")
    RUTA = "/Volumes/workspace/default/clase_26"
elif ENTORNO == "docker":
    RUTA = "/opt/datos/clase_26"
elif os.name == "nt":
    RUTA = "C:/SparkLocal/workspace/datos/clase_26"
else:
    RUTA = str(Path.home() / "spark_datos" / "clase_26")       # por si alguien usa Linux o macOS

Path(RUTA).mkdir(parents=True, exist_ok=True)

print("Entorno:", ENTORNO)
print("Versión de Spark:", spark.version)
print("Carpeta de datos:", RUTA)
if ENTORNO != "databricks":
    print("Master:", spark.sparkContext.master)

NOTE: Picked up JDK_JAVA_OPTIONS: -Duser.home=/home/spark -Dlog4j2.configurationFile=/opt/spark/conf/log4j2.properties --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/jdk.internal.loader=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=tru

Entorno: docker
Versión de Spark: 4.1.4
Carpeta de datos: /opt/datos/clase_26
Master: spark://spark-master:7077


## Sección 1. ¿Qué es Apache Spark?
### Ejemplo 1.1 · La primera celda de Spark

In [2]:
print("Tipo del objeto spark:", type(spark).__name__)
print("¿Spark 4 o superior?", int(spark.version.split(".")[0]) >= 4)

numeros = spark.range(1, 6)
numeros.show()
display(numeros)

Tipo del objeto spark: SparkSession
¿Spark 4 o superior? True


+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
+---+



id
1
2
3
4
5


## Sección 2. Arquitectura de Spark
### Ejemplo 2.1 · Particiones de un millón de transacciones

In [3]:
from pyspark.sql import functions as F

def contar_particiones(df):
    return df.select(F.spark_partition_id().alias("particion")).distinct().count()

transacciones = spark.range(0, 1_000_000, numPartitions=4)
print("Filas:", transacciones.count())
print("Particiones iniciales:", contar_particiones(transacciones))

repartidas = transacciones.repartition(8)
print("Particiones tras repartition(8):", contar_particiones(repartidas))
print("Particiones del DataFrame original:", contar_particiones(transacciones))

Filas: 1000000


Particiones iniciales: 4
Particiones tras repartition(8): 8
Particiones del DataFrame original: 4


### Ejemplo 2.2 · Ver el trabajo de Spark

In [4]:
from pyspark.sql import functions as F

pedidos = spark.range(0, 2_000_000).withColumn("importe", (F.col("id") % 500) * 1.5)

print("Pedidos:", pedidos.count())
print("Pedidos de más de 700 €:", pedidos.filter(F.col("importe") > 700).count())

Pedidos: 2000000
Pedidos de más de 700 €: 132000


## Sección 3. DataFrames y esquemas
### Ejemplo 3.1 · Catálogo de productos de una tienda de electrónica

In [5]:
productos = [
    ("SKU-101", "Auriculares Bluetooth", "Audio", 59.90, 120),
    ("SKU-102", "Teclado mecánico", "Periféricos", 89.00, 45),
    ("SKU-103", "Monitor 27 pulgadas", "Pantallas", 249.99, 18),
    ("SKU-104", "Ratón inalámbrico", "Periféricos", 24.50, 300),
]
columnas = ["sku", "nombre", "categoria", "precio", "stock"]

catalogo = spark.createDataFrame(productos, columnas)

catalogo.printSchema()
catalogo.show()
print("Columnas:", catalogo.columns)
print("Tipos:", catalogo.dtypes)
print("Número de productos:", catalogo.count())

root
 |-- sku: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: long (nullable = true)



+-------+--------------------+-----------+------+-----+
|    sku|              nombre|  categoria|precio|stock|
+-------+--------------------+-----------+------+-----+
|SKU-101|Auriculares Bluet...|      Audio|  59.9|  120|
|SKU-102|    Teclado mecánico|Periféricos|  89.0|   45|
|SKU-103| Monitor 27 pulgadas|  Pantallas|249.99|   18|
|SKU-104|   Ratón inalámbrico|Periféricos|  24.5|  300|
+-------+--------------------+-----------+------+-----+

Columnas: ['sku', 'nombre', 'categoria', 'precio', 'stock']
Tipos: [('sku', 'string'), ('nombre', 'string'), ('categoria', 'string'), ('precio', 'double'), ('stock', 'bigint')]


Número de productos: 4


### Ejemplo 3.2 · Esquema inferido frente a esquema explícito

In [6]:
from datetime import date

from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, DateType)

pedidos = [
    ("PED-5001", "Ferretería Ruiz", 3, 149.50, date(2026, 11, 2)),
    ("PED-5002", "Bodegas Sol", 12, 1320.00, date(2026, 11, 3)),
    ("PED-5003", "Talleres Norte", 1, 89.90, date(2026, 11, 3)),
]

# 1. Esquema inferido
inferido = spark.createDataFrame(pedidos, ["id_pedido", "cliente", "unidades", "importe", "fecha"])
print("--- Esquema inferido ---")
inferido.printSchema()

# 2. Esquema explícito con StructType
esquema = StructType([
    StructField("id_pedido", StringType(), nullable=False),
    StructField("cliente", StringType(), nullable=True),
    StructField("unidades", IntegerType(), nullable=True),
    StructField("importe", DoubleType(), nullable=True),
    StructField("fecha", DateType(), nullable=True),
])
explicito = spark.createDataFrame(pedidos, esquema)
print("--- Esquema con StructType ---")
explicito.printSchema()

# 3. Esquema explícito como texto DDL
ddl = "id_pedido STRING NOT NULL, cliente STRING, unidades INT, importe DOUBLE, fecha DATE"
con_ddl = spark.createDataFrame(pedidos, ddl)
print("--- Esquema con DDL ---")
con_ddl.printSchema()
con_ddl.show()

--- Esquema inferido ---
root
 |-- id_pedido: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- unidades: long (nullable = true)
 |-- importe: double (nullable = true)
 |-- fecha: date (nullable = true)

--- Esquema con StructType ---
root
 |-- id_pedido: string (nullable = false)
 |-- cliente: string (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- importe: double (nullable = true)
 |-- fecha: date (nullable = true)

--- Esquema con DDL ---
root
 |-- id_pedido: string (nullable = false)
 |-- cliente: string (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- importe: double (nullable = true)
 |-- fecha: date (nullable = true)

+---------+---------------+--------+-------+----------+
|id_pedido|        cliente|unidades|importe|     fecha|
+---------+---------------+--------+-------+----------+
| PED-5001|Ferretería Ruiz|       3|  149.5|2026-11-02|
| PED-5002|    Bodegas Sol|      12| 1320.0|2026-11-03|
| PED-5003| Talleres Norte|       1|  

### Ejemplo 3.3 · Opciones de `show()` y valores nulos

In [7]:
incidencias = [
    ("INC-901", "Pago duplicado en la pasarela de tarjetas del cliente", "alta", None),
    ("INC-902", "Retraso en el envío", "media", "Logística"),
    ("INC-903", "No llega el email de confirmación del pedido realizado", None, "Sistemas"),
    ("INC-904", "Producto dañado", "alta", "Calidad"),
]
df = spark.createDataFrame(incidencias, "id STRING, descripcion STRING, prioridad STRING, equipo STRING")

print("1) show() por defecto")
df.show()

print("2) Solo 2 filas y sin recortar el texto")
df.show(2, truncate=False)

print("3) Formato vertical, recortando a 25 caracteres")
df.show(1, truncate=25, vertical=True)

1) show() por defecto
+-------+--------------------+---------+---------+
|     id|         descripcion|prioridad|   equipo|
+-------+--------------------+---------+---------+
|INC-901|Pago duplicado en...|     alta|     NULL|
|INC-902| Retraso en el envío|    media|Logística|
|INC-903|No llega el email...|     NULL| Sistemas|
|INC-904|     Producto dañado|     alta|  Calidad|
+-------+--------------------+---------+---------+

2) Solo 2 filas y sin recortar el texto
+-------+-----------------------------------------------------+---------+---------+
|id     |descripcion                                          |prioridad|equipo   |
+-------+-----------------------------------------------------+---------+---------+
|INC-901|Pago duplicado en la pasarela de tarjetas del cliente|alta     |NULL     |
|INC-902|Retraso en el envío                                  |media    |Logística|
+-------+-----------------------------------------------------+---------+---------+
only showing top 2 rows
3

## Sección 4. Leer ficheros
### Ejemplo 4.1 · Ventas de una cadena de tiendas de café

In [8]:
from pathlib import Path

# 0. Preparar el fichero de datos en RUTA (en un proyecto real ya existiría)
Path(f"{RUTA}/ventas_tiendas.csv").write_text(
    "fecha,tienda,producto,unidades,precio_unitario\n"
    "2026-11-02,Madrid Centro,Cafetera espresso,2,149.90\n"
    "2026-11-02,Valencia Puerto,Molinillo eléctrico,5,39.50\n"
    "2026-11-03,Madrid Centro,Café en grano 1 kg,12,18.75\n"
    "2026-11-03,Sevilla Triana,Cafetera espresso,1,149.90\n",
    encoding="utf-8",
)

# 1. Sin opciones
sin_opciones = spark.read.csv(f"{RUTA}/ventas_tiendas.csv")
print("--- Sin opciones ---")
sin_opciones.printSchema()
sin_opciones.show(2)

# 2. Con cabecera
con_cabecera = spark.read.csv(f"{RUTA}/ventas_tiendas.csv", header=True)
print("--- header=True ---")
con_cabecera.printSchema()

# 3. Con cabecera e inferencia de tipos
ventas = spark.read.csv(f"{RUTA}/ventas_tiendas.csv", header=True, inferSchema=True)
print("--- header=True, inferSchema=True ---")
ventas.printSchema()
ventas.show()

--- Sin opciones ---
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)

+----------+-------------+-----------------+--------+---------------+
|       _c0|          _c1|              _c2|     _c3|            _c4|
+----------+-------------+-----------------+--------+---------------+
|     fecha|       tienda|         producto|unidades|precio_unitario|
|2026-11-02|Madrid Centro|Cafetera espresso|       2|         149.90|
+----------+-------------+-----------------+--------+---------------+
only showing top 2 rows
--- header=True ---
root
 |-- fecha: string (nullable = true)
 |-- tienda: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- unidades: string (nullable = true)
 |-- precio_unitario: string (nullable = true)



--- header=True, inferSchema=True ---
root
 |-- fecha: date (nullable = true)
 |-- tienda: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)

+----------+---------------+-------------------+--------+---------------+
|     fecha|         tienda|           producto|unidades|precio_unitario|
+----------+---------------+-------------------+--------+---------------+
|2026-11-02|  Madrid Centro|  Cafetera espresso|       2|          149.9|
|2026-11-02|Valencia Puerto|Molinillo eléctrico|       5|           39.5|
|2026-11-03|  Madrid Centro| Café en grano 1 kg|      12|          18.75|
|2026-11-03| Sevilla Triana|  Cafetera espresso|       1|          149.9|
+----------+---------------+-------------------+--------+---------------+



### Ejemplo 4.2 · Lecturas de contadores con esquema y registros erróneos

In [9]:
from pathlib import Path

Path(f"{RUTA}/lecturas_contadores.csv").write_text(
    "id_contador;fecha;consumo_kwh;tarifa\n"
    "CNT-0001;2026-10-31;312.4;2.0TD\n"
    "CNT-0002;2026-10-31;error_lectura;2.0TD\n"
    "CNT-0003;2026-10-31;1840.0;3.0TD\n"
    "CNT-0004;31/10/2026;275.9;2.0TD\n"
    "CNT-0005;2026-10-31;98.2;2.0TD\n",
    encoding="utf-8",
)

esquema = "id_contador STRING, fecha DATE, consumo_kwh DOUBLE, tarifa STRING"

# 1. PERMISSIVE (por defecto): los valores que no encajan se convierten en NULL
permisivo = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .schema(esquema)
    .csv(f"{RUTA}/lecturas_contadores.csv")
)
print("--- mode = PERMISSIVE ---")
permisivo.printSchema()
permisivo.show()

# 2. DROPMALFORMED: descarta las filas que no encajan con el esquema
descartando = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("mode", "DROPMALFORMED")
    .schema(esquema)
    .csv(f"{RUTA}/lecturas_contadores.csv")
)
print("--- mode = DROPMALFORMED ---")
descartando.show()

# 3. FAILFAST: se detiene en la primera fila incorrecta
estricto = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .schema(esquema)
    .csv(f"{RUTA}/lecturas_contadores.csv")
)
print("--- mode = FAILFAST ---")
try:
    estricto.show()
except Exception:
    print("Lectura cancelada: hay registros que no cumplen el esquema")

--- mode = PERMISSIVE ---
root
 |-- id_contador: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- consumo_kwh: double (nullable = true)
 |-- tarifa: string (nullable = true)

+-----------+----------+-----------+------+
|id_contador|     fecha|consumo_kwh|tarifa|
+-----------+----------+-----------+------+
|   CNT-0001|2026-10-31|      312.4| 2.0TD|
|   CNT-0002|2026-10-31|       NULL| 2.0TD|
|   CNT-0003|2026-10-31|     1840.0| 3.0TD|
|   CNT-0004|      NULL|      275.9| 2.0TD|
|   CNT-0005|2026-10-31|       98.2| 2.0TD|
+-----------+----------+-----------+------+

--- mode = DROPMALFORMED ---
+-----------+----------+-----------+------+
|id_contador|     fecha|consumo_kwh|tarifa|
+-----------+----------+-----------+------+
|   CNT-0001|2026-10-31|      312.4| 2.0TD|
|   CNT-0003|2026-10-31|     1840.0| 3.0TD|
|   CNT-0005|2026-10-31|       98.2| 2.0TD|
+-----------+----------+-----------+------+

--- mode = FAILFAST ---


Lectura cancelada: hay registros que no cumplen el esquema


26/11/25 18:39:58 ERROR TaskSetManager: Task 0 in stage 55.0 failed 4 times; aborting job


### Ejemplo 4.3 · Telemetría de camiones en JSON con campos anidados

In [10]:
from pathlib import Path

Path(f"{RUTA}/telemetria_camiones.json").write_text(
    '{"camion": "TRK-11", "velocidad_kmh": 87.5, "ubicacion": {"ciudad": "Zaragoza", "lat": 41.65, "lon": -0.88}, "alertas": ["neumatico"]}\n'
    '{"camion": "TRK-12", "velocidad_kmh": 0.0, "ubicacion": {"ciudad": "Bilbao", "lat": 43.26, "lon": -2.93}, "alertas": []}\n'
    '{"camion": "TRK-13", "velocidad_kmh": 104.2, "ubicacion": {"ciudad": "Madrid", "lat": 40.42, "lon": -3.70}, "alertas": ["velocidad", "combustible"]}\n',
    encoding="utf-8",
)

telemetria = spark.read.json(f"{RUTA}/telemetria_camiones.json")
telemetria.printSchema()
telemetria.show(truncate=False)

print("--- Campos del struct anidado ---")
telemetria.select("camion", "ubicacion.ciudad", "ubicacion.lat").show()

root
 |-- alertas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- camion: string (nullable = true)
 |-- ubicacion: struct (nullable = true)
 |    |-- ciudad: string (nullable = true)
 |    |-- lat: double (nullable = true)
 |    |-- lon: double (nullable = true)
 |-- velocidad_kmh: double (nullable = true)

+------------------------+------+------------------------+-------------+
|alertas                 |camion|ubicacion               |velocidad_kmh|
+------------------------+------+------------------------+-------------+
|[neumatico]             |TRK-11|{Zaragoza, 41.65, -0.88}|87.5         |
|[]                      |TRK-12|{Bilbao, 43.26, -2.93}  |0.0          |
|[velocidad, combustible]|TRK-13|{Madrid, 40.42, -3.7}   |104.2        |
+------------------------+------+------------------------+-------------+

--- Campos del struct anidado ---
+------+--------+-----+
|camion|  ciudad|  lat|
+------+--------+-----+
|TRK-11|Zaragoza|41.65|
|TRK-12|  Bilbao|43

## Sección 5. Transformaciones, acciones y evaluación perezosa
### Ejemplo 5.1 · Medir la pereza

In [11]:
import time

from pyspark.sql import functions as F

clics = spark.range(0, 50_000_000).withColumnRenamed("id", "id_clic")

inicio = time.perf_counter()
pares = clics.filter(F.col("id_clic") % 2 == 0)                        # transformación
con_importe = pares.withColumn("importe", F.col("id_clic") * 0.01)     # transformación
grandes = con_importe.filter(F.col("importe") > 400_000)               # transformación
t_transformaciones = time.perf_counter() - inicio

print("Tipo de 'grandes':", type(grandes).__name__)
print("¿Las transformaciones tardaron menos de 1 segundo?", t_transformaciones < 1)

inicio = time.perf_counter()
total = grandes.count()                                                # acción
t_accion = time.perf_counter() - inicio

print("Filas que cumplen el filtro:", total)
print("¿La acción tardó más que las transformaciones?", t_accion > t_transformaciones)

Tipo de 'grandes': DataFrame
¿Las transformaciones tardaron menos de 1 segundo? True
Filas que cumplen el filtro: 4999999
¿La acción tardó más que las transformaciones? True


### Ejemplo 5.2 · Errores de análisis frente a errores de datos

In [12]:
from pyspark.sql import functions as F

facturas = spark.createDataFrame(
    [("FAC-01", "1200"), ("FAC-02", "980"), ("FAC-03", "mil quinientos")],
    "factura STRING, importe_texto STRING",
)

# 1. Error de ANÁLISIS: la columna no existe
try:
    mal_escrita = facturas.select("importe_total")       # PySpark clásico falla AQUÍ
    print("1) select() definido sin errores")
    mal_escrita.columns                                  # Spark Connect falla AQUÍ
except Exception as error:
    print("2) Error de análisis:", type(error).__name__)

# 2. Error de DATOS: la conversión es válida, pero un valor no se puede convertir
convertidas = facturas.withColumn("importe", F.col("importe_texto").cast("int"))
print("3) Columnas de 'convertidas':", convertidas.columns)

try:
    convertidas.show()                                   # aquí se procesan los datos
except Exception:
    print("4) Falla al ejecutar la acción show()")

# 3. Conversión tolerante: try_cast devuelve NULL en lugar de fallar
facturas.withColumn("importe", F.col("importe_texto").try_cast("int")).show()

{"ts": "2026-11-25 18:40:02.228", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `importe_total` cannot be resolved. Did you mean one of the following? [`importe_texto`, `factura`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o271.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `importe_total` cannot be resolved. Did you mean one of the following? [`importe_texto`, `factura`]. SQLSTATE: 42703;\n'Project ['importe_total]\n+- LogicalRDD [factura#385, importe_texto#386], false\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeErro

2) Error de análisis: AnalysisException
3) Columnas de 'convertidas': ['factura', 'importe_texto', 'importe']


26/11/25 18:40:02 ERROR TaskSetManager: Task 1 in stage 63.0 failed 4 times; aborting job
{"ts": "2026-11-25 18:40:02.984", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'mil quinientos' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 17 in cell [12]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o285.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'mil quinientos' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataF

4) Falla al ejecutar la acción show()
+-------+--------------+-------+
|factura| importe_texto|importe|
+-------+--------------+-------+
| FAC-01|          1200|   1200|
| FAC-02|           980|    980|
| FAC-03|mil quinientos|   NULL|
+-------+--------------+-------+



### Ejemplo 5.3 · El plan de ejecución con `explain()`

In [13]:
from pathlib import Path

from pyspark.sql import functions as F

Path(f"{RUTA}/envios.csv").write_text(
    "envio,destino,peso_kg,tarifa\n"
    "ENV-1,Madrid,2.5,4.90\n"
    "ENV-2,Bilbao,12.0,9.50\n"
    "ENV-3,Madrid,31.0,18.00\n"
    "ENV-4,Sevilla,0.8,3.90\n",
    encoding="utf-8",
)

envios = spark.read.csv(
    f"{RUTA}/envios.csv",
    header=True,
    schema="envio STRING, destino STRING, peso_kg DOUBLE, tarifa DOUBLE",
)

resultado = (
    envios
    .filter(F.col("destino") == "Madrid")
    .withColumn("tarifa_con_iva", F.round(F.col("tarifa") * 1.21, 2))
    .filter(F.col("peso_kg") > 1)
    .select("envio", "tarifa_con_iva")
)

resultado.explain()
resultado.show()

== Physical Plan ==
*(1) Project [envio#409, round((tarifa#412 * 1.21), 2) AS tarifa_con_iva#414]
+- *(1) Filter (((isnotnull(destino#410) AND isnotnull(peso_kg#411)) AND (destino#410 = Madrid)) AND (peso_kg#411 > 1.0))
   +- FileScan csv [envio#409,destino#410,peso_kg#411,tarifa#412] Batched: false, DataFilters: [isnotnull(destino#410), isnotnull(peso_kg#411), (destino#410 = Madrid), (peso_kg#411 > 1.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/datos/clase_26/envios.csv], PartitionFilters: [], PushedFilters: [IsNotNull(destino), IsNotNull(peso_kg), EqualTo(destino,Madrid), GreaterThan(peso_kg,1.0)], ReadSchema: struct<envio:string,destino:string,peso_kg:double,tarifa:double>


+-----+--------------+
|envio|tarifa_con_iva|
+-----+--------------+
|ENV-1|          5.93|
|ENV-3|         21.78|
+-----+--------------+



## Sección 6. Trabajar con columnas
### Ejemplo 6.1 · Seleccionar columnas de la plantilla

In [14]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

empleados = spark.createDataFrame(
    [("E-01", "Laura Gómez", "Finanzas", 42000.0, "Madrid"),
     ("E-02", "Iván Torres", "Ventas", 31000.0, "Valencia"),
     ("E-03", "Nuria Sanz", "Tecnología", 51000.0, "Madrid")],
    "id STRING, nombre STRING, departamento STRING, salario_anual DOUBLE, sede STRING",
)

# 1. Seleccionar por nombre de columna (texto)
empleados.select("nombre", "departamento").show()

# 2. Las cuatro formas de referirse a una columna
empleados.select(
    "nombre",                        # texto
    col("sede"),                     # función col()
    empleados.departamento,          # atributo del DataFrame
    empleados["salario_anual"],      # índice del DataFrame
).show()

# 3. Expresiones y alias
empleados.select(
    col("nombre").alias("empleado"),
    F.round(col("salario_anual") / 14, 2).alias("salario_mensual_14_pagas"),
    F.upper(col("sede")).alias("sede_mayus"),
).show()

+-----------+------------+
|     nombre|departamento|
+-----------+------------+
|Laura Gómez|    Finanzas|
|Iván Torres|      Ventas|
| Nuria Sanz|  Tecnología|
+-----------+------------+

+-----------+--------+------------+-------------+
|     nombre|    sede|departamento|salario_anual|
+-----------+--------+------------+-------------+
|Laura Gómez|  Madrid|    Finanzas|      42000.0|
|Iván Torres|Valencia|      Ventas|      31000.0|
| Nuria Sanz|  Madrid|  Tecnología|      51000.0|
+-----------+--------+------------+-------------+



+-----------+------------------------+----------+
|   empleado|salario_mensual_14_pagas|sede_mayus|
+-----------+------------------------+----------+
|Laura Gómez|                  3000.0|    MADRID|
|Iván Torres|                 2214.29|  VALENCIA|
| Nuria Sanz|                 3642.86|    MADRID|
+-----------+------------------------+----------+



### Ejemplo 6.2 · Calcular las líneas de factura de una tienda deportiva

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

lineas = spark.createDataFrame(
    [("PED-1", "Zapatillas trail", 2, 89.90, 0.10),
     ("PED-1", "Calcetines técnicos", 3, 12.50, 0.00),
     ("PED-2", "Mochila 30 L", 1, 64.00, 0.15)],
    "pedido STRING, producto STRING, uds INT, precio DOUBLE, dto DOUBLE",
)

facturado = (
    lineas
    .withColumn("bruto", col("uds") * col("precio"))
    .withColumn("descuento", F.round(col("bruto") * col("dto"), 2))
    .withColumn("base", F.round(col("bruto") - col("descuento"), 2))
    .withColumn("iva", F.round(col("base") * 0.21, 2))
    .withColumn("total", F.round(col("base") + col("iva"), 2))
    .withColumn("moneda", F.lit("EUR"))
    .withColumnRenamed("uds", "unidades")
    .drop("dto", "bruto")
)

facturado.show()
print("Columnas de 'lineas' (original):", lineas.columns)
print("Columnas de 'facturado':", facturado.columns)

+------+-------------------+--------+------+---------+------+-----+-----+------+
|pedido|           producto|unidades|precio|descuento|  base|  iva|total|moneda|
+------+-------------------+--------+------+---------+------+-----+-----+------+
| PED-1|   Zapatillas trail|       2|  89.9|    17.98|161.82|33.98|195.8|   EUR|
| PED-1|Calcetines técnicos|       3|  12.5|      0.0|  37.5| 7.88|45.38|   EUR|
| PED-2|       Mochila 30 L|       1|  64.0|      9.6|  54.4|11.42|65.82|   EUR|
+------+-------------------+--------+------+---------+------+-----+-----+------+

Columnas de 'lineas' (original): ['pedido', 'producto', 'uds', 'precio', 'dto']
Columnas de 'facturado': ['pedido', 'producto', 'unidades', 'precio', 'descuento', 'base', 'iva', 'total', 'moneda']


### Ejemplo 6.3 · Segmentar clientes con `when()` y convertir tipos

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

clientes = spark.createDataFrame(
    [("C-100", "Bodegas Sol", 18500.0, "12"),
     ("C-101", "Talleres Norte", 4200.0, "3"),
     ("C-102", "Ferretería Ruiz", 950.0, "1"),
     ("C-103", "Hostelería Levante", 7600.0, "9")],
    "id STRING, cliente STRING, facturacion_anual DOUBLE, antiguedad_anios STRING",
)

segmentados = (
    clientes
    .withColumn("antiguedad_anios", col("antiguedad_anios").cast("int"))
    .withColumn(
        "segmento",
        F.when(col("facturacion_anual") >= 10000, "Premium")
         .when(col("facturacion_anual") >= 3000, "Estándar")
         .otherwise("Básico"),
    )
    .withColumn("fidelizado", col("antiguedad_anios") >= 5)
)

segmentados.printSchema()
segmentados.show()

root
 |-- id: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- facturacion_anual: double (nullable = true)
 |-- antiguedad_anios: integer (nullable = true)
 |-- segmento: string (nullable = false)
 |-- fidelizado: boolean (nullable = true)

+-----+------------------+-----------------+----------------+--------+----------+
|   id|           cliente|facturacion_anual|antiguedad_anios|segmento|fidelizado|
+-----+------------------+-----------------+----------------+--------+----------+
|C-100|       Bodegas Sol|          18500.0|              12| Premium|      true|
|C-101|    Talleres Norte|           4200.0|               3|Estándar|     false|
|C-102|   Ferretería Ruiz|            950.0|               1|  Básico|     false|
|C-103|Hostelería Levante|           7600.0|               9|Estándar|      true|
+-----+------------------+-----------------+----------------+--------+----------+



## Sección 7. Filtrar, ordenar y quitar duplicados
### Ejemplo 7.1 · Detectar transferencias sospechosas

In [17]:
from pyspark.sql.functions import col

transferencias = spark.createDataFrame(
    [("TR-01", "ES", "DE", 1200.0, "web"),
     ("TR-02", "ES", "ES", 15000.0, "oficina"),
     ("TR-03", "FR", "KY", 9800.0, "app"),
     ("TR-04", "ES", "PA", 450.0, "app"),
     ("TR-05", "IT", "ES", 22000.0, "web"),
     ("TR-06", "ES", "FR", 3100.0, "app")],
    "id STRING, origen STRING, destino STRING, importe DOUBLE, canal STRING",
)

print("1) Importe mayor de 10 000")
transferencias.filter(col("importe") > 10000).show()

print("2) Canal app Y importe mayor de 1 000 (where es un alias de filter)")
transferencias.where((col("canal") == "app") & (col("importe") > 1000)).show()

print("3) Destino en paraísos fiscales O importe mayor de 20 000")
paraisos = ["KY", "PA"]
transferencias.filter(col("destino").isin(paraisos) | (col("importe") > 20000)).show()

print("4) Transferencias internacionales (origen distinto de destino) que NO son de la web")
transferencias.filter((col("origen") != col("destino")) & ~(col("canal") == "web")).show()

print("5) Importe entre 1 000 y 10 000, ambos incluidos")
transferencias.filter(col("importe").between(1000, 10000)).select("id", "importe").show()

1) Importe mayor de 10 000
+-----+------+-------+-------+-------+
|   id|origen|destino|importe|  canal|
+-----+------+-------+-------+-------+
|TR-02|    ES|     ES|15000.0|oficina|
|TR-05|    IT|     ES|22000.0|    web|
+-----+------+-------+-------+-------+

2) Canal app Y importe mayor de 1 000 (where es un alias de filter)
+-----+------+-------+-------+-----+
|   id|origen|destino|importe|canal|
+-----+------+-------+-------+-----+
|TR-03|    FR|     KY| 9800.0|  app|
|TR-06|    ES|     FR| 3100.0|  app|
+-----+------+-------+-------+-----+

3) Destino en paraísos fiscales O importe mayor de 20 000
+-----+------+-------+-------+-----+
|   id|origen|destino|importe|canal|
+-----+------+-------+-------+-----+
|TR-03|    FR|     KY| 9800.0|  app|
|TR-04|    ES|     PA|  450.0|  app|
|TR-05|    IT|     ES|22000.0|  web|
+-----+------+-------+-------+-----+

4) Transferencias internacionales (origen distinto de destino) que NO son de la web
+-----+------+-------+-------+-----+
|   id|o

### Ejemplo 7.2 · Leads comerciales con datos incompletos

In [18]:
from pyspark.sql.functions import col

leads = spark.createDataFrame(
    [("L-1", "ana@bodegasol.es", "600111222", "Feria"),
     ("L-2", None, "611222333", "Web"),
     ("L-3", "luis@talleresnorte.es", None, None),
     ("L-4", None, None, "Web")],
    "id STRING, email STRING, telefono STRING, origen STRING",
)

print("1) Leads sin email")
leads.filter(col("email").isNull()).show()

print("2) Leads contactables: email O teléfono")
leads.filter(col("email").isNotNull() | col("telefono").isNotNull()).show()

print("3) ¡Cuidado! Comparar con == None no funciona")
print("   Filas con email == None:", leads.filter(col("email") == None).count())
print("   Filas con email.isNull():", leads.filter(col("email").isNull()).count())

print("4) Un filtro con != también descarta los NULL")
print("   origen != 'Web':", [fila.id for fila in leads.filter(col("origen") != "Web").collect()])

1) Leads sin email
+---+-----+---------+------+
| id|email| telefono|origen|
+---+-----+---------+------+
|L-2| NULL|611222333|   Web|
|L-4| NULL|     NULL|   Web|
+---+-----+---------+------+

2) Leads contactables: email O teléfono
+---+--------------------+---------+------+
| id|               email| telefono|origen|
+---+--------------------+---------+------+
|L-1|    ana@bodegasol.es|600111222| Feria|
|L-2|                NULL|611222333|   Web|
|L-3|luis@talleresnort...|     NULL|  NULL|
+---+--------------------+---------+------+

3) ¡Cuidado! Comparar con == None no funciona
   Filas con email == None: 0
   Filas con email.isNull(): 2
4) Un filtro con != también descarta los NULL
   origen != 'Web': ['L-1']


### Ejemplo 7.3 · Ordenar y deduplicar reservas de hotel

In [19]:
from pyspark.sql.functions import col

reservas = spark.createDataFrame(
    [("R-1", "Hotel Mar Azul", "Málaga", 3, 360.0),
     ("R-2", "Hotel Sierra", "Granada", 2, 210.0),
     ("R-3", "Hotel Mar Azul", "Málaga", 3, 360.0),
     ("R-4", "Hostal Centro", "Madrid", 1, 75.0),
     ("R-5", "Hotel Sierra", "Granada", 5, 540.0),
     ("R-6", "Hostal Centro", "Madrid", 2, 150.0)],
    "reserva STRING, hotel STRING, ciudad STRING, noches INT, importe DOUBLE",
)

print("1) Ordenar por ciudad (asc) y por importe (desc)")
reservas.orderBy(col("ciudad").asc(), col("importe").desc(), col("reserva")).show()

print("2) Las 3 reservas de mayor importe")
reservas.orderBy(col("importe").desc(), col("reserva")).limit(3).show()

print("3) Hoteles distintos")
reservas.select("hotel", "ciudad").distinct().orderBy("hotel").show()

print("4) Quitar duplicados según algunas columnas")
sin_dup = reservas.dropDuplicates(["hotel", "noches", "importe"])
print("   Filas antes:", reservas.count(), "| después:", sin_dup.count())

1) Ordenar por ciudad (asc) y por importe (desc)
+-------+--------------+-------+------+-------+
|reserva|         hotel| ciudad|noches|importe|
+-------+--------------+-------+------+-------+
|    R-5|  Hotel Sierra|Granada|     5|  540.0|
|    R-2|  Hotel Sierra|Granada|     2|  210.0|
|    R-6| Hostal Centro| Madrid|     2|  150.0|
|    R-4| Hostal Centro| Madrid|     1|   75.0|
|    R-1|Hotel Mar Azul| Málaga|     3|  360.0|
|    R-3|Hotel Mar Azul| Málaga|     3|  360.0|
+-------+--------------+-------+------+-------+

2) Las 3 reservas de mayor importe
+-------+--------------+-------+------+-------+
|reserva|         hotel| ciudad|noches|importe|
+-------+--------------+-------+------+-------+
|    R-5|  Hotel Sierra|Granada|     5|  540.0|
|    R-1|Hotel Mar Azul| Málaga|     3|  360.0|
|    R-3|Hotel Mar Azul| Málaga|     3|  360.0|
+-------+--------------+-------+------+-------+

3) Hoteles distintos
+--------------+-------+
|         hotel| ciudad|
+--------------+-------+
| 

## Sección 8. Acciones: traer resultados al programa
### Ejemplo 8.1 · Alertas críticas de una monitorización de servidores

In [20]:
from pyspark.sql.functions import col

alertas = spark.createDataFrame(
    [("SRV-01", "cpu", 97.5), ("SRV-02", "memoria", 91.0),
     ("SRV-03", "disco", 88.2), ("SRV-04", "cpu", 99.1)],
    "servidor STRING, metrica STRING, valor DOUBLE",
)
criticas = alertas.filter(col("valor") > 90).orderBy(col("valor").desc())

print("count():", criticas.count())

primera = criticas.first()
print("first():", primera)
print("   Acceso por nombre:", primera.servidor, "| por clave:", primera["valor"],
      "| por posición:", primera[1])

print("take(2):", criticas.take(2))

filas = criticas.collect()
print("collect() devuelve una", type(filas).__name__, "de", len(filas), "elementos")
servidores = [fila.servidor for fila in filas]
print("Servidores a revisar:", servidores)
print("Fila como diccionario:", filas[0].asDict())

count(): 3
first(): Row(servidor='SRV-04', metrica='cpu', valor=99.1)
   Acceso por nombre: SRV-04 | por clave: 99.1 | por posición: cpu
take(2): [Row(servidor='SRV-04', metrica='cpu', valor=99.1), Row(servidor='SRV-01', metrica='cpu', valor=97.5)]
collect() devuelve una list de 3 elementos
Servidores a revisar: ['SRV-04', 'SRV-01', 'SRV-02']
Fila como diccionario: {'servidor': 'SRV-04', 'metrica': 'cpu', 'valor': 99.1}


## Sección 9. Un pipeline completo con PySpark
### Ejemplo 9.1 · Limpieza de las entregas diarias de una empresa de reparto

In [21]:
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.functions import col

# 0. Fichero de origen: exportación nocturna del sistema de reparto
Path(f"{RUTA}/entregas_2026_11_10.csv").write_text(
    "id_entrega,repartidor,ciudad,peso_kg,minutos_entrega,estado\n"
    "ENT-001, marta díaz ,MADRID,2.4,35,entregado\n"
    "ENT-002,jorge pons,barcelona,15.0,82,ENTREGADO\n"
    "ENT-003,marta díaz,madrid,,40,entregado\n"
    "ENT-004,ana ruiz,Valencia,0.9,n/d,incidencia\n"
    "ENT-005,jorge pons,Barcelona,7.5,58,entregado\n"
    "ENT-006,ana ruiz,valencia,32.0,121,Entregado\n"
    "ENT-002,jorge pons,barcelona,15.0,82,ENTREGADO\n"
    "ENT-007,luis gil,Sevilla,4.2,47,devuelto\n",
    encoding="utf-8",
)

# 1. EXTRACCIÓN: todo como texto para que ninguna fila se pierda al leer
crudo = spark.read.csv(f"{RUTA}/entregas_2026_11_10.csv", header=True)
print("Filas leídas:", crudo.count())

# 2. TRANSFORMACIÓN
limpio = (
    crudo
    .dropDuplicates(["id_entrega"])
    .withColumn("repartidor", F.initcap(F.trim(col("repartidor"))))
    .withColumn("ciudad", F.initcap(F.trim(col("ciudad"))))
    .withColumn("estado", F.lower(F.trim(col("estado"))))
    .withColumn("peso_kg", col("peso_kg").try_cast("double"))
    .withColumn("minutos_entrega", col("minutos_entrega").try_cast("int"))
)

validas = limpio.filter(
    (col("estado") == "entregado")
    & col("peso_kg").isNotNull()
    & col("minutos_entrega").isNotNull()
)

resultado = (
    validas
    .withColumn(
        "tipo_paquete",
        F.when(col("peso_kg") < 5, "pequeño")
         .when(col("peso_kg") < 20, "mediano")
         .otherwise("grande"),
    )
    .withColumn("a_tiempo", col("minutos_entrega") <= 60)
    .select("id_entrega", "repartidor", "ciudad", "peso_kg",
            "tipo_paquete", "minutos_entrega", "a_tiempo")
    .orderBy("ciudad", "id_entrega")
)

# 3. RESULTADO (en una clase posterior se guardará en una tabla)
print("Filas tras quitar duplicados:", limpio.count())
print("Entregas válidas:", resultado.count())
resultado.show()

retrasadas = [fila.id_entrega for fila in resultado.filter(~col("a_tiempo")).collect()]
print("Entregas fuera de plazo:", retrasadas)

Filas leídas: 8
Filas tras quitar duplicados: 7
Entregas válidas: 4
+----------+----------+---------+-------+------------+---------------+--------+
|id_entrega|repartidor|   ciudad|peso_kg|tipo_paquete|minutos_entrega|a_tiempo|
+----------+----------+---------+-------+------------+---------------+--------+
|   ENT-002|Jorge Pons|Barcelona|   15.0|     mediano|             82|   false|
|   ENT-005|Jorge Pons|Barcelona|    7.5|     mediano|             58|    true|
|   ENT-001|Marta Díaz|   Madrid|    2.4|     pequeño|             35|    true|
|   ENT-006|  Ana Ruiz| Valencia|   32.0|      grande|            121|   false|
+----------+----------+---------+-------+------------+---------------+--------+

Entregas fuera de plazo: ['ENT-002', 'ENT-006']


### Cerrar la sesión
Ejecuta esta celda al terminar la Clase 26 para liberar los núcleos del clúster antes de abrir otro notebook.

In [22]:
if ENTORNO != "databricks":
    spark.stop()